# CNN Visual Grader (MobileNetV2 Experimentation)
**Objective:** Fine-tune a pre-trained MobileNetV2 model to predict an engineered, log-scaled sales target based solely on product images. 

**Professional Standards Implemented:**
* Target distribution EDA.
* 80/20 `train_test_split`.
* Data augmentation to prevent overfitting.
* Early stopping and model checkpointing to preserve the best weights.

## Setup and Imports

In [4]:
# Imports:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [5]:
# Ensuring GPU Memory Growth:
gpu = tf.config.experimental.list_physical_devices('GPU')

if gpu:
    try:
        for g in gpu:
            tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError as e:
        print(e)

print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(gpu)}")

TensorFlow version: 2.20.0
Num GPUs Available: 1


## Data Loading and Validation

In [16]:
# Data Directory Paths:
IMAGE_DIR= os.path.abspath('../data/images_10k/')
TARGET_CSV= os.path.abspath('../data/processed/cnn_proxy_targets_10k.csv')

In [17]:
# Loading Data:
df= pd.read_csv(TARGET_CSV,
                dtype= {'article_id': str})
df['file_path']= df['article_id'].apply(lambda x: os.path.join(IMAGE_DIR, f"{x}.jpg"))

In [18]:
df.head()

,article_id,total_sales,total_sales_log,cnn_target_scaled,file_path
0,0111593001,13888,9.538852,0.873014,/home/shail/trendsight/data/images_10k/0111593...
1,0112679048,3,1.386294,0.068409,/home/shail/trendsight/data/images_10k/0112679...
2,0114428030,101,4.624973,0.388046,/home/shail/trendsight/data/images_10k/0114428...
3,0118458004,230,5.442418,0.468722,/home/shail/trendsight/data/images_10k/0118458...
4,0118458028,35,3.583519,0.285261,/home/shail/trendsight/data/images_10k/0118458...


In [20]:
df.shape

(10000, 5)

In [22]:
# Verifying Image Existence:
df['image_exists']= df['file_path'].apply(os.path.exists)
missing_count= (~df['image_exists']).sum()
print(f"Missing Images Dropped: {missing_count}")
df= df[df['image_exists']].copy()

print(f"DataFrame Shape After Dropping Missing Images: {df.shape}")

Missing Images Dropped: 39
DataFrame Shape After Dropping Missing Images: (9961, 6)
